# Model Performance

In [27]:
import pandas as pd
from pathlib import Path

path_dir = Path("../data/processed/predictions")

global_predictions = pd.read_csv(path_dir / "global_predictions.csv")

demographic_datasets = {
    "sex": ["female", "male"],
    "age": ["younger_adult", "middle_age", "older"],
    "income": ["lower_income", "middle_income", "higher_income"],
    "education": ["below_high_school", "high_school", "some_college_or_higher"]
}

demographic_predictions = {}

for category, groups in demographic_datasets.items():
    demographic_predictions[category] = {}

    for group in groups:
        demographic_predictions[category][group] = pd.read_csv(
            path_dir / f"{category}_{group}_predictions.csv"
        )

global_group_predictions = {}

for category, groups in demographic_datasets.items():
    global_group_predictions[category] = {}

    for group in groups:
        global_group_predictions[category][group] = pd.read_csv(
            path_dir / f"{category}_{group}_global_calibrated_predictions.csv"
        )

In [30]:
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

def evaluate_model(y_true, y_pred):
    return {
        "brier_score": brier_score_loss(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_pred),
        "ece": calculate_ece(y_true.to_numpy(), y_pred.to_numpy())
    }

global_results = pd.DataFrame([
    {"model": column, **evaluate_model(global_predictions["y_true"], global_predictions[column])}
    for column in global_predictions.columns[1:]
])

global_results

,model,brier_score,log_loss,roc_auc,ece
0,log_reg_uncalibrated,0.099372,0.320983,0.820088,0.012032
1,log_reg_sigmoid,0.099367,0.320983,0.820088,0.012123
2,log_reg_isotonic,0.098903,0.324024,0.819721,0.004302
3,gb_uncalibrated,0.097180,0.314709,0.827049,0.003945
4,gb_sigmoid,0.097206,0.314705,0.827049,0.004099
5,gb_isotonic,0.097252,0.315324,0.826816,0.004510


In [31]:
demographic_results = []

for category, groups in demographic_predictions.items():
    for group, predictions in groups.items():
        for column in predictions.columns[1:]:
            demographic_results.append({
                "category": category,
                "group": group,
                "model": column,
                **evaluate_model(predictions["y_true"], predictions[column])
            })

demographic_results = pd.DataFrame(demographic_results)

demographic_results

,category,group,model,brier_score,log_loss,roc_auc,ece
0,sex,female,log_reg_uncalibrated,0.092341,0.300543,0.831208,0.010223
1,sex,female,log_reg_sigmoid,0.092363,0.300556,0.831208,0.010225
2,sex,female,log_reg_isotonic,0.091910,0.305346,0.830716,0.004952
3,sex,female,gb_uncalibrated,0.090366,0.294628,0.837362,0.005083
4,sex,female,gb_sigmoid,0.090389,0.294637,0.837362,0.004343
...,...,...,...,...,...,...,...
61,education,some_college_or_higher,log_reg_sigmoid,0.085851,0.283849,0.826191,0.009685
62,education,some_college_or_higher,log_reg_isotonic,0.085427,0.286261,0.825643,0.007391
63,education,some_college_or_higher,gb_uncalibrated,0.083657,0.277638,0.833161,0.002569
64,education,some_college_or_higher,gb_sigmoid,0.083746,0.277757,0.833161,0.005449


In [32]:
global_group_results = []

for category, groups in global_group_predictions.items():
    for group, predictions in groups.items():
        for column in predictions.columns[1:]:
            global_group_results.append({
                "category": category,
                "group": group,
                "model": column,
                **evaluate_model(predictions["y_true"], predictions[column])
            })

global_group_results = pd.DataFrame(global_group_results)

global_group_results

,category,group,model,brier_score,log_loss,roc_auc,ece
0,sex,female,log_reg_uncalibrated,0.092542,0.300894,0.831108,0.012381
1,sex,female,log_reg_sigmoid,0.092598,0.300916,0.831108,0.011992
2,sex,female,log_reg_isotonic,0.092073,0.304392,0.830818,0.005138
3,sex,female,gb_uncalibrated,0.090448,0.295058,0.837100,0.005516
4,sex,female,gb_sigmoid,0.090501,0.295016,0.837100,0.005354
...,...,...,...,...,...,...,...
61,education,some_college_or_higher,log_reg_sigmoid,0.085852,0.284005,0.826106,0.009867
62,education,some_college_or_higher,log_reg_isotonic,0.085388,0.287940,0.825345,0.007700
63,education,some_college_or_higher,gb_uncalibrated,0.083557,0.277693,0.833096,0.002735
64,education,some_college_or_higher,gb_sigmoid,0.083692,0.277782,0.833096,0.005280
